# Argus — plate detector training (Kaggle GPU)

Trains the single-class Indian license-plate detector. This is the **primary**
training path; the local CPU fallback exists only if Kaggle is unavailable.

**Before running:**

1. Settings → Accelerator → **GPU T4 x2** (or P100).
2. Settings → Internet → **On** (needed to `pip install ultralytics` and clone).
3. Add Data → search an Indian number plate dataset in **YOLO format** and
   attach it. Roboflow Universe exports work directly. It lands under
   `/kaggle/input/<slug>/`.
4. Set `SRC` in the next cell to that path.

Runtime: ~20-40 s/epoch, so 50 epochs is roughly 20-35 minutes. Kaggle allows
12 h per session and 30 h/week, so there is a lot of headroom — but sessions can
die, which is what the resume cell at the bottom is for.


In [ ]:
SRC     = "/kaggle/input/CHANGE-ME"   # <- attached dataset root
EPOCHS  = 50
SUBSET  = 3000    # train images; raise toward 15000, GPU can afford it
VAL     = 400

In [ ]:
!nvidia-smi
!pip -q install ultralytics

In [ ]:
# The repo carries prepare_dataset.py and train_plate.py — no code duplicated here.
%cd /kaggle/working
!rm -rf Argus && git clone -q https://github.com/Deeptanshu789/Argus.git
%cd /kaggle/working/Argus

## Prepare

`prepare_dataset.py` finds every image/label pair at any depth, drops unlabelled
images, splits, and writes a valid `data.yaml`. It handles both flat and
pre-split layouts, so it does not matter which of the two shapes the attached
dataset uses.

In [ ]:
!python ml/prepare_dataset.py --src "{SRC}" --subset {SUBSET} --val {VAL}
!cat datasets/plates/data.yaml

## Train

GPU defaults: `imgsz=640, batch=32, amp=True, freeze=0`. Note `freeze=0` — the
backbone is *not* frozen here. Freezing it is a CPU concession that costs
accuracy; on a T4 there is no reason to pay it.

**Go/no-go bar: `mAP50 >= 0.85` on val.** Single-class, tight-boxed plates reach
this readily. Below 0.7 means the dataset conversion is wrong, not the training —
check that labels are class `0` and boxes are normalized `xywh`.

In [ ]:
!python ml/train_plate.py --epochs {EPOCHS} --device 0

In [ ]:
from IPython.display import Image, display
R = "runs/detect/plate"
display(Image(f"{R}/results.png"))
display(Image(f"{R}/confusion_matrix_normalized.png"))

## Export the weights

Download `argus-plate-weights.zip` from the notebook's Output panel, then in the
repo:

```bash
mkdir -p runs/detect/plate/weights
unzip argus-plate-weights.zip -d runs/detect/plate/weights
python ml/export_onnx.py --weights runs/detect/plate/weights/best.pt
```

`export_onnx.py` produces OpenVINO int8, which is what the demo machine runs —
the *inference* box is still the CPU laptop even though training moved to a GPU.

In [ ]:
!cd runs/detect/plate/weights && zip -q /kaggle/working/argus-plate-weights.zip best.pt last.pt
!ls -lh /kaggle/working/argus-plate-weights.zip

## If the session died mid-run

Kaggle sessions can be killed. Re-run the clone and prepare cells, then:

In [ ]:
# !python ml/train_plate.py --epochs {EPOCHS} --device 0 --resume